# 3. Structured Output (Advanced) — 제품 탑재형 AI: 기기 제어/진단 요청서 파싱
**Pydantic 스키마 + Cross-field Validation + Discriminated Union + Repair(재생성) 전략**

---

## 🧩 시나리오(가상)
제품에 탑재된 LLM이 사용자의 자연어를 바로 실행하면 위험합니다.  
대신, **반드시 구조화된 JSON(명령/진단/질문)** 으로 변환한 뒤에

- 안전 체크(확인/권한)
- 실행 가능한 API/SDK 호출
- 로그/근거 저장

같은 파이프라인을 거칩니다.

이 노트북은 그중 핵심인 **"자연어 → 스키마(JSON)"** 를 튼튼하게 만드는 실습입니다.


In [ ]:
!pip -q install -U langchain langchain-openai pydantic

In [1]:
import os, getpass
from typing import Literal, Optional, List, Union
from datetime import date

from pydantic import BaseModel, Field, model_validator
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


PROXY_URL = "http://10.0.1.5:8000/v1" 
PROXY_TOKEN = "67abdcf53fa16de5d91983a4b8e30a62" 


model = init_chat_model(os.getenv("OPENAI_MODEL", "gpt-4o-mini"), base_url=PROXY_URL, api_key=PROXY_TOKEN)


/opt/conda/envs/sam/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) 스키마 설계: 기기 제어(Device Control) (Cross-field Validation 포함)

In [2]:
class DeviceControlRequest(BaseModel):
    intent: Literal["device_control"] = "device_control"
    device_type: Literal["robot_vacuum","air_conditioner","washing_machine","refrigerator"]
    device_id: Optional[str] = Field(default=None, description="기기 ID(예: DEV-2002). 없으면 None")
    action: Literal[
        "start", "stop",
        "set_mode",
        "set_temperature",
        "dock",
        "factory_reset",
    ]
    mode: Optional[str] = None
    target_temp_c: Optional[int] = None
    confirmed: bool = False  # 위험 작업 확인(예: factory_reset)

    @model_validator(mode="after")
    def _check_rules(self):
        # device_id가 없으면 실행 불가 → 보통 다음 단계에서 질문을 생성하도록 처리
        # (여기서는 스키마 검증 자체는 통과시키되, 파이프라인에서 추가 질문을 하게 만들 수 있습니다.)

        if self.action == "set_temperature":
            if self.device_type != "air_conditioner":
                raise ValueError("set_temperature는 air_conditioner에서만 가능합니다.")
            if self.target_temp_c is None:
                raise ValueError("set_temperature에는 target_temp_c가 필요합니다.")
            if not (16 <= self.target_temp_c <= 30):
                raise ValueError("target_temp_c는 16~30 범위여야 합니다.")
        if self.action == "set_mode":
            if self.mode is None:
                raise ValueError("set_mode에는 mode가 필요합니다.")
        if self.action == "dock" and self.device_type != "robot_vacuum":
            raise ValueError("dock은 robot_vacuum에서만 가능합니다.")
        if self.action == "factory_reset":
            if not self.confirmed:
                raise ValueError("factory_reset은 confirmed=true(사용자 확인) 후에만 가능합니다.")
        return self


## 2) 스키마 설계: 고장 진단(Troubleshooting)

In [3]:
class TroubleshootRequest(BaseModel):
    intent: Literal["troubleshooting"] = "troubleshooting"
    device_type: Literal["robot_vacuum","air_conditioner","washing_machine","refrigerator"]
    device_id: Optional[str] = Field(default=None, description="기기 ID(예: DEV-2002). 없으면 None")
    symptom: str = Field(description="사용자 증상 요약")
    error_code: Optional[str] = Field(default=None, description="표시된 에러코드(없으면 None)")
    safety_risk: bool = Field(default=False, description="연기/과열/가스냄새 등 안전 이슈 여부")
    suggested_steps: List[str] = Field(default_factory=list, description="사용자가 해볼 수 있는 점검/조치 단계")
    needs_service: bool = Field(default=False, description="원격/기사 방문이 필요할 가능성")

    @model_validator(mode="after")
    def _check_rules(self):
        if self.safety_risk:
            # 안전 이슈는 조치보다 우선 안내가 필요 → 최소 1개 단계는 포함
            if len(self.suggested_steps) == 0:
                raise ValueError("safety_risk=true이면 suggested_steps에 최소 1개(전원차단/환기 등)가 필요합니다.")
        if self.error_code is not None and len(self.error_code.strip()) < 2:
            raise ValueError("error_code가 있으면 2자 이상이어야 합니다.")
        return self


## 3) Discriminated Union: intent 자동 선택

In [4]:
RequestUnion = Union[DeviceControlRequest, TroubleshootRequest]

## 4) 에러 복구(Repair) 전략: 검증 실패 → 오류 메시지로 재생성

In [5]:
def error_feedback(e: Exception) -> str:
    return (
        "출력이 스키마 검증에 실패했습니다. 아래 오류를 해결하도록 JSON을 다시 생성하세요.\n"
        f"[ValidationError]\n{str(e)}\n\n"
        "규칙: JSON만 출력, enum은 허용값만 사용, 불확실하면 device_id/error_code는 null로 두세요."
    )

agent = create_agent(
    model=model,
    tools=[],
    system_prompt=(
        "당신은 '제품 탑재형 AI' 요청 파서입니다. 사용자의 자연어를 스키마에 맞는 JSON으로 변환하세요.\n"
        "- intent는 device_control 또는 troubleshooting 중 하나를 선택하세요.\n"
        "- 위험 작업(factory_reset 등)은 confirmed=true 없이 실행하지 않도록 스키마 검증을 만족시켜야 합니다.\n"
        "- 불확실하면 추측하지 말고 device_id/error_code는 null로 두세요.\n"
    ),
    response_format=ToolStrategy(RequestUnion, handle_errors=error_feedback),
)


## 5) 실행 예시: 정상 케이스 / 오류 케이스 (제품 시나리오)

In [6]:
tests = [
    # 기기 제어
    "에어컨 온도 24도로 맞춰줘. 우리 거실 에어컨이야.",
    "로봇청소기 거실 청소 시작해줘",
    # 진단
    "세탁기 5C 에러가 떠요. 물이 안 빠져요.",
    "냉장고에서 탄 냄새가 나고 연기가 나는 것 같아요. 어떻게 해야 하나요?",
    # 위험 작업(확인 없이) → Repair 유도
    "로봇청소기 공장초기화 해줘",
]

for t in tests:
    res = agent.invoke({"messages": [{"role": "user", "content": t}]})
    print("\n---\nINPUT:", t)
    print("OUTPUT:", res["structured_response"])



---
INPUT: 에어컨 온도 24도로 맞춰줘. 우리 거실 에어컨이야.
OUTPUT: intent='device_control' device_type='air_conditioner' device_id=None action='set_temperature' mode=None target_temp_c=24 confirmed=False

---
INPUT: 로봇청소기 거실 청소 시작해줘
OUTPUT: intent='device_control' device_type='robot_vacuum' device_id=None action='start' mode=None target_temp_c=None confirmed=False

---
INPUT: 세탁기 5C 에러가 떠요. 물이 안 빠져요.
OUTPUT: intent='troubleshooting' device_type='washing_machine' device_id=None symptom='물이 안 빠져요' error_code='5C' safety_risk=False suggested_steps=['세탁기 필터를 점검하고 청소해 보세요.', '배수 호스가 꼬이지 않았는지 확인해 보세요.'] needs_service=True

---
INPUT: 냉장고에서 탄 냄새가 나고 연기가 나는 것 같아요. 어떻게 해야 하나요?
OUTPUT: intent='troubleshooting' device_type='refrigerator' device_id=None symptom='탄 냄새가 나고 연기가 남' error_code=None safety_risk=True suggested_steps=['전원 차단', '환기'] needs_service=True

---
INPUT: 로봇청소기 공장초기화 해줘
OUTPUT: intent='troubleshooting' device_type='robot_vacuum' device_id=None symptom='공장 초기화를 요청함' error_code=None safety_risk=Fals

## 📝 실습 과제 (난이도 업) — 제품 탑재형 관점

In [ ]:
# ✅ TODO 셀 (학생용)
# 1) RequestUnion에 'manual_info' intent를 추가해보세요.
#    - 예: 필터 청소 주기, 절전 모드 설명 등
# 2) DeviceControlRequest에 'schedule' (예약 실행) 액션을 추가하고,
#    - start_time이 미래여야 한다 같은 cross-field validation을 넣어보세요.
# 3) troubleshooting에서 safety_risk=true인 경우,
#    - needs_service=true가 자동으로 되도록 validator를 확장해보세요.

raise NotImplementedError("학생용 TODO 셀입니다. 아래 '참고 답안' 보기 전까지 구현해보세요.")


In [7]:
# ✅ 참고 답안

from datetime import datetime

class ManualInfoRequest(BaseModel):
    intent: Literal["manual_info"] = "manual_info"
    device_type: Literal["robot_vacuum","air_conditioner","washing_machine","refrigerator"]
    question: str

class DeviceControlRequestV2(DeviceControlRequest):
    action: Literal[
        "start", "stop",
        "set_mode",
        "set_temperature",
        "dock",
        "factory_reset",
        "schedule",
    ]
    schedule_time: Optional[str] = None  # "YYYY-MM-DD HH:MM"

    @model_validator(mode="after")
    def _check_schedule(self):
        if self.action == "schedule":
            if not self.schedule_time:
                raise ValueError("schedule에는 schedule_time이 필요합니다.")
            try:
                dt = datetime.strptime(self.schedule_time, "%Y-%m-%d %H:%M")
            except Exception:
                raise ValueError("schedule_time 포맷은 YYYY-MM-DD HH:MM 이어야 합니다.")
            if dt <= datetime.now():
                raise ValueError("schedule_time은 현재 시각 이후여야 합니다.")
        return self

RequestUnionV2 = Union[DeviceControlRequestV2, TroubleshootRequest, ManualInfoRequest]

agent_v2 = create_agent(
    model=model,
    tools=[],
    system_prompt=(
        "당신은 제품 탑재형 AI 요청 파서입니다. 사용자의 자연어를 스키마(JSON)로 변환하세요.\n"
        "- intent는 device_control/troubleshooting/manual_info 중 선택\n"
        "- 위험 작업은 confirmed=true 없이는 실행하지 말 것\n"
        "- schedule_time은 YYYY-MM-DD HH:MM\n"
    ),
    response_format=ToolStrategy(RequestUnionV2, handle_errors=error_feedback),
)


# with_structured_output은 Union 타입을 직접 받지 못함
# → 각 intent별 Pydantic 모델로 직접 파싱하는 방식으로 우회

import json as _json
from datetime import datetime

_INTENT_MODEL_MAP = {
    "device_control": DeviceControlRequestV2,
    "troubleshooting": TroubleshootRequest,
    "manual_info": ManualInfoRequest,
}

def parse_request_v2(user_text: str, retries: int = 2):
    today_str = datetime.now().strftime("%Y-%m-%d %H:%M")
    prompt = (
        f"오늘 날짜/시각: {today_str}\n"
        "사용자의 자연어를 분석해 아래 규칙에 따라 JSON 하나만 출력하세요. (설명 금지)\n"
        "- intent: device_control / troubleshooting / manual_info\n"
        "- schedule_time은 YYYY-MM-DD HH:MM, 반드시 현재 시각 이후\n"
        f"\n[요청] {user_text}"
    )
    last_err = None
    for i in range(retries + 1):
        resp = model.invoke(prompt)
        txt = resp.content.strip()
        # 마크다운 코드블록 제거
        if txt.startswith("```"):
            lines = txt.splitlines()
            txt = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])
        try:
            data = _json.loads(txt)
            intent = data.get("intent")
            model_cls = _INTENT_MODEL_MAP.get(intent)
            if model_cls is None:
                raise ValueError(f"알 수 없는 intent: {intent!r}. device_control/troubleshooting/manual_info 중 하나여야 합니다.")
            return model_cls(**data)
        except Exception as e:
            last_err = e
            prompt += f"\n\n[오류] {e}\n위 오류를 해결한 JSON을 다시 출력하세요."
    raise RuntimeError(f"파싱 실패({retries}회 재시도): {last_err}")

demo = "로봇청소기 내일 아침 9시에 청소 예약해줘"
print(parse_request_v2(demo))


intent='device_control' device_type='robot_vacuum' device_id=None action='schedule' mode=None target_temp_c=None confirmed=False schedule_time='2026-03-17 09:00'
